# FPL Squad Optimizer - Dashboard

This notebook uses the refactored `fpl_engine` modules to run the optimization pipeline. 
It is designed to run in Google Colab or a local Jupyter environment.

In [ ]:
# Run this if you are in Google Colab to mount your drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import sys
    # Replace with your actual path in Drive
    sys.path.append('/content/drive/My Drive/Hobby/FPL')
except ImportError:
    pass  # Not running in Colab

In [ ]:
%load_ext autoreload
%autoreload 2
import asyncio
import pandas as pd
import nest_asyncio
nest_asyncio.apply()

from fpl_engine.config import LOCKED_PARAMS, MY_FPL_ID, current_realizable_value_dict
from fpl_engine.data import (
    get_current_players_df, fetch_raw_history_cache,
    get_team_df, get_pos_constraint_df,
    get_current_gameweek, get_max_finished_gameweek,
    get_fpl_gameweek_data, get_my_player_ids,
    get_dynamic_weights,
)
from fpl_engine.features import (
    compute_rolling_team_ratings, blend_team_ratings,
    get_fixture_players_stats_df, compute_global_z_distributions,
)
from fpl_engine.scoring import (
    _fit_bonus_multinomial, _fit_regression_params,
    _calculate_performance_indices, create_optimized_custom_score,
)
from fpl_engine.solver import plan_sequential_transfers, create_optimal_fpl_team
# Note: _render_sync_plots is a diagnostic called internally by
# create_optimized_custom_score(visualize=True); no need to import directly.

## 1. Fetch Data & Build Features

In [ ]:
# 1. Fetch Base Data
current_gw = get_current_gameweek()
data_gameweek = get_max_finished_gameweek()
fpl_team_df = get_team_df()
players_df = get_current_players_df()

print(f"Current Gameweek: {current_gw}")
print(f"Max Finished GW:  {data_gameweek}")

# 2. Fetch Match History (Async)
active_player_ids = players_df[players_df['minutes'] > 0]['id'].tolist()
raw_history_df = asyncio.get_event_loop().run_until_complete(
    fetch_raw_history_cache(active_player_ids, use_cache=True)
)

In [ ]:
# 3. Compute Team Ratings & Blends
team_fixture_df, latest_ratings = compute_rolling_team_ratings(raw_history_df)
blended_team_df, latest_blended_ratings = blend_team_ratings(
    team_fixture_df, latest_ratings, fpl_team_df
)

# 4. Generate Fixture Specific Projections
global_dists = compute_global_z_distributions(blended_team_df)

params = LOCKED_PARAMS.copy()

fixture_player_df = get_fixture_players_stats_df(
    params, raw_history_df, global_dists,
    team_ratings_df=blended_team_df,
    latest_team_ratings=latest_blended_ratings,
)

## 2. Generate Scoring Projections

In [ ]:
# 1. Fit Bonus Multinomial Model
bonus_model = _fit_bonus_multinomial(raw_history_df)

# 2. Fit Regression Weights & Merge into Params
regression_weights = _fit_regression_params(fixture_player_df)
params.update(regression_weights)

# 3. Calculate Final Performance Indices (Expected Points)
gw_projection_df = _calculate_performance_indices(
    fixture_player_df, params, bonus_model
)

# 4. Add Custom Dynamic Score (Optional)
# Requires dynamic weights — compute them first:
# try:
#     fpl_gameweek_data = get_fpl_gameweek_data(MY_FPL_ID)
#     weights = get_dynamic_weights(fpl_gameweek_data, data_gameweek)
#     gw_projection_df = create_optimized_custom_score(
#         gw_projection_df,
#         differential_weight=weights['diff_weight'],
#         upside_weight=weights['upside_weight'],
#         visualize=True,
#     )
# except Exception as e:
#     print(f"Custom score skipped: {e}")

## 3. Run Optimization & Visualizations

In [ ]:
# Optimize Team Selection (uses total_points from bootstrap — no projection needed)
starter_df, bench_df = create_optimal_fpl_team(budget=100)

In [ ]:
# Optional: Sequential Transfer Planner
# Requires custom_score and ceiling_score columns from create_optimized_custom_score.
# Uncomment the custom score block above first, then use:
#
# MY_CURRENT_TEAM_IDS = get_my_player_ids(MY_FPL_ID, current_gw)
# bank_values = 1.1
# locked_values = sum(current_realizable_value_dict.values()) + bank_values
#
# prob, result = plan_sequential_transfers(
#     gw_projection_df,
#     start_gameweek=current_gw + 1,
#     current_team_ids=MY_CURRENT_TEAM_IDS,
#     current_realizable_value_dict=current_realizable_value_dict,
#     bank_balance=bank_values,
#     planning_horizon=3,
#     return_model=True,
# )